# 2.1 The agent loop

Follows [01-agent-loop.md](../01-agent-loop.md). We build a PydanticAI agent that makes one
keyframe image for an ad brief, then open up the loop the framework runs for us.

Set `MODE` first:

- `"fake"`: you play the LLM with a Python function, the image tool draws a placeholder. No network, no credits.
- `"fake-gpu"`: a real LLM on Token Factory (asks for your API key once), placeholder image tool. Costs a few cents of tokens.

In [ ]:
MODE = "fake"  # "fake" or "fake-gpu"

In [ ]:
import sys
from pathlib import Path

MODULE = "02-multi-agent-media-workflows"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "tools.py").exists())
sys.path.insert(0, str(CODE))
ASSETS = CODE.parent / "assets"

In [ ]:
from fake import block_real_models

if MODE == "fake":
    block_real_models()  # any request to a real LLM now fails

## Start with the finished result

A saved run of the module's finished pipeline: brief, keyframe, video, trace and cost per step.
This lesson builds its first piece.

In [ ]:
from tracing import read_events, summarize

reference = ASSETS / "reference-run" / "events.jsonl"
if reference.exists():
    events = read_events(reference)
    for e in events[:5]:
        print(f"{e.kind:14} {e.name:16} {e.workload:9} ${e.cost_usd:.4f}")
    print(summarize(events))
else:
    print("No saved reference run in assets/ yet.")

## The brief

The input for the whole module: one `Brief`. Homework 2 uses the same fields.

In [ ]:
from models import Brief

brief = Brief.from_yaml(ASSETS / "brief.yaml")
brief

## A minimal agent

### The tool

A placeholder until lesson 2.3: it draws the prompt on a flat image and writes one trace event.

In [ ]:
from pydantic_ai import Agent, RunContext

from fake import placeholder_png
from tracing import Tracer, new_run_id


async def generate_image(ctx: RunContext[Tracer], prompt: str, seed: int = 7) -> str:
    """Generate one keyframe image for the ad and return its file path."""
    path = ctx.deps.folder / f"keyframe-seed{seed}.png"
    path.write_bytes(placeholder_png(prompt, 832, 480, seed))
    ctx.deps.emit("tool_call", "generate_image", call_id=ctx.tool_call_id, asset_ref=str(path))
    return str(path)

### Be the LLM

The model is a plain function: it gets the conversation and the tools on offer, and returns
the next message. Ours asks for one tool call, then answers. The agent runs it unchanged.

In [ ]:
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart
from pydantic_ai.models.function import AgentInfo, FunctionModel


def be_the_llm(messages: list, info: AgentInfo) -> ModelResponse:
    """A hand-written model: call generate_image, then report the result."""
    print(f"Model sees {len(messages)} messages, tools: {[t.name for t in info.function_tools]}")
    last = messages[-1].parts[-1]
    if last.part_kind == "tool-return":
        return ModelResponse(parts=[TextPart(f"Keyframe ready: {last.content}")])
    return ModelResponse(parts=[ToolCallPart("generate_image", {"prompt": brief.shot, "seed": 7})])


def make_agent(model) -> Agent:
    return Agent(
        model,
        deps_type=Tracer,
        tools=[generate_image],
        instructions="You make one keyframe image for a video ad brief. Call generate_image once.",
    )


agent = make_agent(FunctionModel(be_the_llm))
prompt = f"Brief: {brief.model_dump_json()}"
tracer = Tracer(new_run_id())
result = await agent.run(prompt, deps=tracer)
print(result.output)

### Swap in the real LLM

One line changes: the model. Now the LLM picks the arguments.

In [ ]:
from tools import token_factory_model

if MODE == "fake-gpu":
    agent = make_agent(token_factory_model())  # model ID pinned in models.py
    result = await agent.run(prompt, deps=tracer)
    print(result.output)

### What goes over the wire

Every turn resends the tool schemas and the whole history, so input tokens grow per step.
With `MODE = "fake"` we print the message history; with `"fake-gpu"` the real second request.

In [ ]:
for message in result.all_messages():
    print(type(message).__name__, [part.part_kind for part in message.parts])

In [ ]:
if MODE == "fake-gpu":
    import json

    import httpx
    from openai import AsyncOpenAI
    from pydantic_ai.models.openai import OpenAIChatModel
    from pydantic_ai.providers.openai import OpenAIProvider

    from models import LLM
    from tools import settings

    sent = []

    async def keep(request: httpx.Request) -> None:
        sent.append(json.loads(request.content))

    client = AsyncOpenAI(
        base_url=LLM.base_url,
        api_key=settings("NEBIUS_API_KEY", secret=("NEBIUS_API_KEY",))["NEBIUS_API_KEY"],
        http_client=httpx.AsyncClient(event_hooks={"request": [keep]}),
    )
    spy = make_agent(OpenAIChatModel(LLM.model, provider=OpenAIProvider(openai_client=client)))
    await spy.run(prompt, deps=tracer)
    second = sent[1]
    print(json.dumps(second["tools"], indent=1)[:1200])
    for m in second["messages"]:
        print(m["role"], str(m.get("content"))[:70], "+ tool_calls" if m.get("tool_calls") else "")

## Step through a run with agent.iter()

`agent.run()` hides how many requests it made. `agent.iter()` yields each node of the run graph.

In [ ]:
async with agent.iter(prompt, deps=tracer) as run:
    async for node in run:
        print(type(node).__name__)
print(run.result.output)

## Stop conditions

A model that never stops asking for images. Limits live in code: `UsageLimits` stops the run
before the fourth tool call. These limits count LLM usage only; the GPU budget comes in 2.5.

In [ ]:
from pydantic_ai import UsageLimitExceeded, UsageLimits


def never_stops(messages: list, info: AgentInfo) -> ModelResponse:
    """Ask for another image every turn, with a new seed."""
    return ModelResponse(
        parts=[ToolCallPart("generate_image", {"prompt": brief.shot, "seed": len(messages)})]
    )


try:
    await make_agent(FunctionModel(never_stops)).run(
        prompt, deps=tracer, usage_limits=UsageLimits(tool_calls_limit=3)
    )
except UsageLimitExceeded as exc:
    print("Stopped:", exc)

## Where state lives

The conversation lives in memory and dies with the process. Save it to the run folder; a new
process continues from the file.

In [ ]:
from tracing import load_messages, save_messages

path = save_messages(tracer.folder, result.all_messages())
print(path, path.stat().st_size, "bytes")

restored = load_messages(tracer.folder)
again = await agent.run("Make one more with seed 8.", message_history=restored, deps=tracer)
print(again.output)

## Trace events

One event per model request and tool call in `events.jsonl`, linked by `call_id`.
`run_traced()` runs the agent node by node and writes the model requests; the tool writes its own.

In [ ]:
from check_trace import check
from tracing import run_traced

tracer = Tracer(new_run_id())
await run_traced(agent, prompt, tracer=tracer, deps=tracer)
for e in tracer.events():
    print(
        f"{e.kind:14} {e.name:16} call_id={e.call_id} parent={(e.parent_id or '-')[:8]} tokens_in={e.tokens_in}"
    )
print("Contract problems:", check(tracer.events()))
print(summarize(tracer.events()))